# 01 Functions: Domain, Range, Composition, Inverse, and Piecewise Rules

| | |
|---|---|
| **Course** | Calculus I |
| **Prerequisite** | Algebra of expressions and solving quadratic equations |

Calculus is the study of change, and change is always measured on a function.
Before a limit or a derivative, you need to say on which numbers the function is defined,
which values it produces, and whether there is a way back.

## Course convention

Every example has two layers, and this order stays fixed.

1. **Hand solution.** The problem is solved by hand: the definition, the algebra, and the result.
2. **Python recheck.** The same result is computed again with SymPy.
   If the simplified difference is zero, the two methods agree.

The output of every code cell is saved in this notebook. You do not have to guess in order to see the answer,
but write the hand solution once yourself before you read that output.

## Definitions

A function $f$ is a rule that assigns exactly one output to each allowed input.
The set of allowed inputs is the **domain**, and the set of outputs that actually occur is the **range**.

Composition applies the inner function first:

$$
(f\circ g)(x)=f(g(x)).
$$

The inverse $f^{-1}$ exists only when $f$ is one-to-one:
each output comes from at most one input.
The derivative test, made precise in lesson 07, is this.
If $f'(x)>0$ on an interval, then $f$ is strictly increasing there, and therefore one-to-one.


In [1]:
import sympy as sp

# Compare a computed SymPy result with the answer found by hand.
def check(found, hand, label="Result"):
    # Simplify each side first so equivalent algebraic forms can match.
    found_s = sp.simplify(found)
    hand_s = sp.simplify(hand)
    difference = sp.simplify(found_s - hand_s)
    print(label)
    print("  Hand:", hand_s)
    print("  Python:", found_s)
    print("  Difference:", difference)
    # The methods agree only when that difference simplifies to zero.
    assert difference == 0
    print("  Confirmed")


## Example 1 — Domain of a radical quotient

### Hand solution

$$
f(x)=\dfrac{\sqrt{x-2}}{x-5}.
$$

The expression under the radical must be nonnegative, so $x-2\ge 0$, that is, $x\ge 2$.
The denominator must not be zero, so $x\neq 5$.
The domain is

$$
[2,5)\cup(5,\infty).
$$

### Python recheck


In [2]:
import sympy as sp

# Build f(x) = sqrt(x - 2) / (x - 5) on the real line.
x = sp.symbols("x", real=True)
f = sp.sqrt(x - 2) / (x - 5)
# Read the domain as the set where this expression is continuous.
domain = sp.calculus.util.continuous_domain(f, x, sp.S.Reals)
print("Domain:", domain)
# Hand answer: [2, 5) union (5, infinity).
assert domain == sp.Union(sp.Interval.Ropen(2, 5), sp.Interval.open(5, sp.oo))
print("Confirmed")


Domain: Union(Interval.Ropen(2, 5), Interval.open(5, oo))
Confirmed


## Example 2 — Composition, and a domain hidden by the look of $f(g(x))$

### Hand solution

$$
g(x)=\sqrt{x},\qquad f(t)=t^2-1.
$$

The composition simplifies, but only on the domain of $g$:

$$
(f\circ g)(x)=x-1,\qquad x\ge 0.
$$

Simplifying the expression does not erase the domain restriction that comes from $g$.
At $x=4$, apply $g$ first: $g(4)=2$, and then $f(2)=3$.

### Python recheck


In [3]:
import sympy as sp

# Composition (f o g)(x) with g(x) = sqrt(x) and f(t) = t^2 - 1.
# Hand answer: the simplified expression is x - 1, but only for x >= 0.
x = sp.symbols("x", real=True)
g = sp.sqrt(x)
composed = sp.simplify(g**2 - 1)
print("Simplified expression:", composed)
print("Value at 4:", (sp.sqrt(4)**2 - 1))
domain = sp.calculus.util.continuous_domain(g**2 - 1, x, sp.S.Reals)
# The square root is still present in the original expression; read the domain from g.
domain_g = sp.calculus.util.continuous_domain(g, x, sp.S.Reals)
print("Domain of g:", domain_g)
# The simplified expression equals x - 1.
assert composed == x - 1
# The domain of g is [0, infinity).
assert domain_g == sp.Interval(0, sp.oo)
# g(4) = 2, then f(2) = 3.
assert sp.sqrt(4)**2 - 1 == 3
print("Confirmed")


Simplified expression: x - 1
Value at 4: 3
Domain of g: Interval(0, oo)
Confirmed


## Example 3 — Building an inverse

### Hand solution

$$
f(x)=\dfrac{2x+1}{x-3},\qquad x\neq 3.
$$

Solve $y=f(x)$ for $x$:

$$
\begin{aligned}
y &= \dfrac{2x+1}{x-3} \\
y(x-3) &= 2x+1 \\
xy-3y &= 2x+1 \\
xy-2x &= 3y+1 \\
x(y-2) &= 3y+1 \\
x &= \dfrac{3y+1}{y-2}.
\end{aligned}
$$

Rename the variable:

$$
f^{-1}(x)=\dfrac{3x+1}{x-2}.
$$

As a check, $f(5)=\dfrac{11}{2}$. The inverse must give $5$ back:

$$
\begin{aligned}
f^{-1}\!\left(\dfrac{11}{2}\right)
&= \dfrac{3\cdot\dfrac{11}{2}+1}{\dfrac{11}{2}-2} \\
&= \dfrac{\dfrac{35}{2}}{\dfrac{7}{2}} \\
&= 5.
\end{aligned}
$$

### Python recheck


In [4]:
import sympy as sp

# Solve y = (2x + 1)/(x - 3) for x.
# Hand inverse, after renaming y to x: (3x + 1)/(x - 2).
x, y = sp.symbols("x y")
equation = sp.Eq(y, (2*x + 1)/(x - 3))
solved = sp.solve(equation, x)[0]
inverse = sp.simplify(solved.subs(y, x))
print("Inverse:", inverse)
# f(5) = 11/2, so the inverse must send 11/2 back to 5.
back = sp.simplify(inverse.subs(x, sp.Rational(11, 2)))
print("Return from 11/2:", back)
assert inverse == (3*x + 1)/(x - 2)
assert back == 5
print("Confirmed")


Inverse: (3*x + 1)/(x - 2)
Return from 11/2: 5
Confirmed


## Example 4 — A piecewise function

### Hand solution

$$
h(x)=\begin{cases}
x^2 & x<1 \\
2x-1 & x\ge 1
\end{cases}
$$

At the joint, the value comes from the second rule: $h(1)=1$.
The left-hand limit is $1^2=1$, and the right-hand limit is $1$ as well, so the graph does not jump at $x=1$.
Lesson 05 will name this continuity. Here we only read the two rules at the boundary point.

One input gives one output: $h(-2)=4$ and $h(3)=5$.

### Python recheck


In [5]:
import sympy as sp

# Piecewise rule: x^2 when x < 1, and 2x - 1 when x >= 1.
# Hand values: h(-2) = 4, h(1) = 1, h(3) = 5.
# Hand limits: both one-sided limits at x = 1 equal 1.
x = sp.symbols("x")
h = sp.Piecewise((x**2, x < 1), (2*x - 1, x >= 1))
print("h(-2) =", h.subs(x, -2))
print("h(1) =", h.subs(x, 1))
print("h(3) =", h.subs(x, 3))
print("Left-hand limit:", sp.limit(h, x, 1, dir="-"))
print("Right-hand limit:", sp.limit(h, x, 1, dir="+"))
assert h.subs(x, -2) == 4
assert h.subs(x, 1) == 1
assert h.subs(x, 3) == 5
assert sp.limit(h, x, 1, dir="-") == sp.limit(h, x, 1, dir="+") == 1
print("Confirmed")


h(-2) = 4
h(1) = 1
h(3) = 5
Left-hand limit: 1
Right-hand limit: 1
Confirmed


## Example 5 — One-to-one behavior, before building an inverse

### Hand solution

$$
f(x)=x^3+x.
$$

$$
\begin{aligned}
f'(x) &= 3x^2+1 \\
&\ge 1 \\
&> 0.
\end{aligned}
$$

The derivative is never zero and never negative, so $f$ is strictly increasing on $\mathbb{R}$ and has an inverse.
The inverse itself is not written with a simple radical. Existence does not require a closed formula.

Since $f(1)=2$, we have $f^{-1}(2)=1$.

### Python recheck


In [6]:
import sympy as sp

# f(x) = x^3 + x. Hand derivative: 3x^2 + 1, whose minimum on the reals is 1.
x = sp.symbols("x", real=True)
f = x**3 + x
derivative = sp.diff(f, x)
print("Derivative:", derivative)
print("Minimum of the derivative:", sp.minimum(derivative, x))
# The derivative matches 3x^2 + 1.
assert sp.simplify(derivative - (3*x**2 + 1)) == 0
# That derivative is always at least 1, so f is strictly increasing.
assert sp.minimum(derivative, x) == 1
# f(1) = 2, which is why the inverse sends 2 back to 1.
assert f.subs(x, 1) == 2
print("Confirmed")


Derivative: 3*x**2 + 1
Minimum of the derivative: 1
Confirmed


## A pitfall

A zero denominator punches a hole in the domain even when that factor disappears after simplification.
A zero under an even root cuts the domain off.
The simplified expression and the original function agree only on the original domain.

## Summary

Write the domain before every limit and every derivative.
Evaluate a composition from the inside outward.
Build an inverse by solving $y=f(x)$ for $x$, and be sure $f$ is one-to-one before you start.
